# COS711 Assignment 2: Solar Filament Segmentation

Initial Google Colab setup. This notebook downloads and inspects the official Kaggle competition files only.

## 1. Google colab check

In [ ]:
#GPU test
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

# Check NVIDIA GPU availability through PyTorch
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU count:", torch.cuda.device_count())
    else:
        print("No GPU allocated to this runtime.")

except ImportError:
    print("PyTorch is not installed.")

## 2. Get data from shared drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/Shareddrives/COS 711/Assignment 2/filament-segmentation-2026.zip" -d /content/data

## 3. Inspect dataset file names

In [ ]:
from pathlib import Path

path = "/content/data/filament-segmentation-2026"
data_root = Path(path)
for item in sorted(data_root.rglob('*')):
    if item.is_file():
        print(item.relative_to(data_root))

## Data Preprocessing

The following steps are used:
1. Parse and group: load COCO Json and decode polygons
2. Split into train and validation tests 85/15
3. Convert to working resolution
4. Normalisation, 1st-99th percentiles are cliped and all data is scaled to [0, 1]
5. Build targets per annotated image

In [ ]:
!pip -q install pycocotools opencv-python-headless

In [ ]:
import json, re, math
import numpy as np, pandas as pd, cv2
from pycocotools import mask as mu
from tqdm.auto import tqdm
from joblib import Parallel, delayed

DATA = Path("/content/data/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
TRAIN_IMG = DATA / "train/train_images"
ANN_JSON = next((DATA / "train").glob("*.json"))
CACHE = Path("/content/cache")

### Step 1

In [ ]:
coco = json.load(open(ANN_JSON))
FNAME_RE = re.compile(r"^(\d{14})([A-Za-z]{2})")

imgs = pd.DataFrame(coco["images"])
imgs["image_id"] = imgs["id"].astype(str)
imgs["stem"] = imgs.file_name.map(lambda s: Path(s).stem)

m = imgs.stem.str.extract(FNAME_RE)
assert m.notna().all().all(), "Some file names do not match the expected pattern."

imgs["ts"], imgs["inst"] = m[0], m[1]
imgs["obs"] = imgs.ts + "_" + imgs.inst
imgs["annotator"] = imgs.image_id.str.split("-").str[0]
imgs["date"] = pd.to_datetime(imgs.ts, format="%Y%m%d%H%M%S")

on_disk = {p.stem: p for p in TRAIN_IMG.iterdir()}
imgs["path"] = imgs.stem.map(on_disk)
assert imgs.path.notna().all(), imgs[imgs.path.isna()].file_name.head()

# Merge all 4 categories into one class
ann_by_img = {}
for a in coco["annotations"]:
    ann_by_img.setdefault(a["image_id"], []).append(a)

missing = set(ann_by_img) - set(imgs.image_id)
print("Annotation image_ids no in images:", len(missing), missing)
print(pd.Series([a["category_id"] for a in coco["annotations"]]).value_counts())

obs = (imgs.groupby("obs")
       .agg(ts=("ts", "first"), inst=("inst", "first"), date=("date", "first"),
            stem=("stem", "first"), path=("path", "first"),
            n_sets=("image_id", "nunique")).reset_index())

print(len(imgs), "annotated image sets ->", len(obs), "observations")
print(obs.inst.value_counts())

### Step 2: Split

In [ ]:
BLOCK_DAYS, VAL_SPLIT, SEED = 7, 0.15, 1

obs["block"] = (obs.date - obs.date.min()).dt.days // BLOCK_DAYS

def pick_val_blocks(obs, val_frac, seed=SEED):
    rng = np.random.default_rng(seed)
    total = obs.inst.value_counts()
    target = total * val_frac

    block = (obs.groupby(["block", "inst"]).size().unstack(fill_value=0)
              .reindex(columns=total.index, fill_value=0))
    cur, chosen = pd.Series(0.0, index=total.index), []

    for b in rng.permutation(block.index.values):
        new = cur + block.loc[b]
        if (new > target * 1.5 + 2).any():
            continue

        chosen.append(b)
        cur = new

        if (cur >= 0.9 * target).all() or cur.sum() >= target.sum():
            break

    return chosen

val_blocks = pick_val_blocks(obs, VAL_SPLIT)
obs["split"] = np.where(obs.block.isin(val_blocks), "val", "train")
print(pd.crosstab(obs.inst, obs.split, margins=True))

#Test and train dates test
test_ts = [re.match(FNAME_RE, p.stem).group(1) for p in (DATA/"test/test_images").glob("*.jp*g")]
test_dates = pd.to_datetime(pd.Series(test_ts), format="%Y%m%d%H%M%S")
print("train:", obs.date.min(), "->", obs.date.max(), " test:", test_dates.min(), "->", test_dates.max())


### Step 3: Verify resolution and downsample

In [ ]:
def raster_crop(a, scale, h, w, pad=2):
    "Raterise one polygon into a bbox crop"
    polys = [np.asarray(p, float).reshape(-1, 2) * scale for p in a["segmentation"]]
    all_points = np.concatenate(polys)

    x0 = max(int(np.floor(all_points[:, 0].min())) - pad, 0)
    y0 = max(int(np.floor(all_points[:, 1].min())) - pad, 0)

    x1 = min(int(np.ceil(all_points[:, 0].max())) + pad + 1, w)
    y1 = min(int(np.ceil(all_points[:, 1].max())) + pad + 1, h)

    if x1 <= x0 or y1 <= y0:
        raise None

    seg = [(p - [x0, y0]).ravel().tolist() for p in polys]
    mask = mu.decode(mu.merge(mu.frPyObjects(seg, y1 - y0, x1 - x0)))
    return np.ascontiguousarray(mask), x0, y0, x1, y1

def spine_points(a):
    sp = np.asarray(a.get("spine") or [], float).ravel()
    return sp[: sp.size // 2 * 2].reshape(-1, 2)

rows = []
for a in tqdm(coco["annotations"]):
    r = raster_crop(a, 1.0, 2048, 2048)

    if r is None: continue

    m = r[0]
    d = cv2.distanceTransform(m, cv2.DIST_L2, 5)
    sp = spine_points(a)
    L = np.linalg.norm(np.diff(sp, axis=0), axis=1).sum() if len(sp) > 1 else np.nan

    rows.append(dict(area=m.sum(), spine_len=L, mean_w=m.sum() / L if L > 0 else np.nan, max_w=2*d.max()))

geo = pd.DataFrame(rows)
print("Geometric stats for", len(geo), "annotations")
print(geo.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(1))
print("Fraction with mean width < 4 px:", (geo.mean_w < 4).mean().round(3))
print("Fraction with max width < 4 px:", (geo.max_w < 4).mean().round(3))

### Step 4: Normalisation

In [ ]:
DISK_MARGIN = 1.01

def circle_mask(h, w, cx, cy, r):
    "Create a circular mask of radius r at (cx, cy) in an image of size h x w"
    yy, xx = np.ogrid[:h, :w]
    return (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2

def fit_disk(gray):
    "Otsu on a downsampled image to find largest blob and fit a circle on its contour"
    h, w = gray.shape
    s = cv2.GaussianBlur(cv2.resize(gray, (512, 512), interpolation=cv2.INTER_AREA), (0,0), 2)

    t, _ = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats((s > t).astype(np.uint8))

    if n < 2: return None

    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    cnts, _ = cv2.findContours((lab == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    points = max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(float)

    A = np.c_[2 * points, np.ones(len(points))]
    (cx, cy, c), *_ = np.linalg.lstsq(A, (points ** 2).sum(1), rcond=None)
    r = math.sqrt(c + cx ** 2 + cy ** 2)

    rms = np.sqrt(np.mean((np.hypot(points[:, 0]-cx, points[:,1]-cy) -r) ** 2)) / r
    touches_border = (points.min() <= 1) or (points.max() >= 510)

    # Cannot fit
    if not(0.2 * 512 < r < 0.7 * 512) or rms > 0.03: return None

    # return dict(otsu=t, n_blobs=n-1, blob_area_frac=blob.mean(), r=r, r_frac=r/512,
    #             rms=rms, touches_border=touches_border, cx=cx, cy=cy), blob

    return cx * w / 512, cy * h / 512, r * w / 512

# bad = obs_meta[~obs_meta.disk_ok].obs
# rows = []
# for o in bad:
#     path = obs.set_index("obs").path[o]
#     gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
#     d, blob = fit_disk(gray)
#     d["obs"] = o
#     rows.append(d)
# print(pd.DataFrame(rows).round(3))

In [ ]:
def normalise(gray, disk):
    lo, hi = np.percentile(gray[disk], [1,99])
    x = np.clip((gray.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
    x[~disk] = 0
    return (x * 255 + 0.5).astype(np.uint8), lo, hi

### Step 5: Build targets

In [ ]:
def build_targets(anns, h, w, scale, spine_r=1, contact_r=1):
    inst = np.zeros((h, w), np.uint16)
    dist = np.zeros((h, w), np.float32)
    spine = np.zeros((h, w), np.uint8)
    centers = []

    for k, a in enumerate(sorted(anns, key=lambda a: -a["area"]), 1):
        r = raster_crop(a, scale, h, w)

        if r is None: continue

        m, x0, y0, x1, y1 = r
        mb = m.astype(bool)

        if not mb.any(): continue

        inst[y0:y1, x0:x1][mb] = k
        d = cv2.distanceTransform(m, cv2.DIST_L2, 5)
        d /= max(d.max(), 1e-6)
        dist[y0:y1, x0:x1][mb] = d[mb]
        ys, xs = np.nonzero(mb)

        centers.append([round(float(xs.mean() + x0), 1), round(float(ys.mean() + y0), 1)])
        sp = spine_points(a) * scale

        if len(sp) > 1:
            cv2.polylines(spine, [np.round(sp).astype(np.int32).reshape(-1, 1, 2)], False, 1, 1)

    if spine_r > 0:
        spine = cv2.dilate(spine, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*spine_r+1,)*2))

    # boundary and contact
    lab = inst.astype(np.float32)
    fg = lab > 0
    k3 = np.ones((3, 3), np.uint8)
    boundary = fg & ((cv2.dilate(lab, k3) != lab) | (cv2.erode(lab, k3) != lab))
    kc = np.ones((2*contact_r+1,)*2, np.uint8)

    mx = cv2.dilate(lab, kc)
    mn = cv2.erode(np.where(fg, lab, np.float32(1e9)), kc)
    contact = (mx > 0) & (mn < 1e9) & (mx != mn)
    contact = cv2.dilate(contact.astype(np.uint8), k3).astype(bool)

    boundary |= contact

    target = np.stack([fg, boundary, contact, np.round(dist * 255), spine]).astype(np.uint8)
    return inst, target, centers

In [ ]:
def process_obs(obs_key, path, inst_name, set_ids, ann_by_img, scale, out):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    H0, W0 = gray.shape
    H, W = int(round(H0 * scale)), int(round(W0 * scale))

    p = fit_disk(gray)
    disk_ok = p is not None

    cx, cy, r = p if disk_ok else (W0 / 2, H0 /2, min(H0, W0) / 2)

    disk0 = circle_mask(H0, W0, cx, cy, r * DISK_MARGIN)
    raw = gray[disk0].astype(np.float32)
    img8, lo, hi = normalise(gray, disk0)

    if scale != 1:
        img8 = cv2.resize(img8, (W, H), interpolation=cv2.INTER_AREA)

    cx, cy, r = cx * scale, cy * scale, r * scale
    disk = circle_mask(H, W, cx, cy, r * DISK_MARGIN)
    img8[~disk] = 0

    soft = np.zeros((H, W), np.float32)
    set_rows = []
    for iid in set_ids:
        anns = ann_by_img.get(iid, [])
        inst, tgt, centers = build_targets(anns, H, W, scale)
        np.save(out / "tgt"/ f"{iid}.npy", tgt)
        np.save(out / "inst" / f"{iid}.npy", inst)
        soft += tgt[0]

        set_rows.append(dict(image_id=iid, obs=obs_key, n_inst=len(centers), centers=json.dumps(centers)))

    soft = (soft / max(len(set_ids), 1) * 255 + 0.5).astype(np.uint8)
    np.save(out / "obs" / f"{obs_key}.npy", np.stack([img8, disk.astype(np.uint8), soft]))

    nrm = img8[disk].astype(np.float32) / 255
    obs_row = dict(obs=obs_key, inst=inst_name, cx=cx, cy=cy, r=r, disk_ok=disk_ok, 
                   raw_mean=raw.mean(), raw_std=raw.std(), raw_p01=lo, raw_p99=hi, 
                   norm_mean=nrm.mean(), norm_std=nrm.std())
    return obs_row, set_rows

In [ ]:
def build_cache(scale=1.0):
    out = CACHE / f"scale{scale}"

    for d in ("obs", "tgt", "inst"): (out / d).mkdir(parents=True, exist_ok=True)
    sets = imgs.groupby("obs").image_id.apply(list)
    jobs = (delayed(process_obs)(r.obs, r.path, r.inst, sets[r.obs], ann_by_img, scale, out)
            for r in obs.itertuples())

    res = list(tqdm(Parallel(n_jobs=-1, return_as="generator")(jobs), total=len(obs)))
    obs_meta = pd.DataFrame([r[0] for r in res]).merge(obs[["obs", "date", "split", "n_sets"]], on="obs")
    set_meta = pd.DataFrame([s for r in res for s in r[1]]).merge(obs_meta[["obs", "inst", "split"]], on="obs")
    obs_meta.to_csv(out / "obs_meta.csv", index=False)
    set_meta.to_csv(out / "set_meta.csv", index=False)
    return obs_meta, set_meta

obs_meta, set_meta = build_cache(scale=1.0)
print("failed disk fits:", (~obs_meta.disk_ok).sum(), "of", len(obs_meta))
print(obs_meta.groupby("inst")[["raw_mean","raw_std","raw_p01","raw_p99","norm_mean","norm_std"]]
      .agg(["mean","std"]).round(2).T)

In [ ]:
print(obs_meta[~obs_meta.disk_ok].inst.value_counts())

### Setup dataset for the U-net

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class FilamentDS(Dataset):
    def __init__(self, set_meta, obs_meta, cache, crop=512, train=True, pos_prob=0.8, limb_w=0.3):
        self.m = set_meta.merge(obs_meta[["obs", "cx", "cy", "r"]], on="obs").reset_index(drop=True)
        self.c, self.crop, self.train, self.pos_prob, self.limn_w = Path(cache), crop, train, pos_prob, limb_w

    def __len__(self): return len(self.w)

    def _origin(self, r, H, W, rng):
        cs = self.crop
        centers = json.loads(r.centers)
        if centers and rng.random() < self.pos_prob:
            cx, cy = centers[rng.integer(len(centers))]
            x0 = int(cx - cs / 2 + rng.integers(-cs // 4, cs // 4 + 1))
            y0 = int(cy - cs / 2 + rng.integers(-cs // 4, cs // 4 + 1))
            
        else:
            lx, ly = int(np.clip(r.cx - r.r, 0, W - cs)), int(np.clip(r.cy - r.r, 0, H - cs))
            hx, hy = int(np.clip(r.cx + r.r - cs, lx, W - cs)), int(np.clip(r.cy + r.r - cs, ly, H-cs))
            x0, y0 = rng.integers(lx, hx + 1), rng.integers(ly, hy + 1)

        return int(np.clip(y0, 0, H - cs)), int(np.clip(x0, 0, W - cs))

    def __getitem__(self, i):
        r = self.m.iloc[i]
        # (3,H,W): img, disk, soft
        obs = np.load(self.c / "obs" / f"{r.obs}.npy", nmap_node="r")

        # (5,H,W)
        tgt = np.load(self.c / "tgt" / f"{r.image_id}.npy", nmap_mode="r")
        H, W = tgt.shape[1:]
        rng = np.random.default_rng()

        if self.train:
            y0, x0 = self._origin(r, H, W, rng)
            h = w = self.crop

        else:
            y0 = x0 = 0
            h, w = H, W

        o = np.array(obs[:, y0:y0+h, x0:x0+w])
        t = np.array(tgt[:, y0:y0+h, x0:x0+w])

        yy, xx = np.mgrid[y0:y0+h, x0:x0+w]
        rn = np.hypot(xx - r.cx, yy - r.cy) / r.r
        wt = np.interp(rn, [0.9, 1.0], [1.0, self.limb_w]).astype(np.float32) * o[1]

        arrs = [o, t, wt[None]]

        if self.train:
            k, flip = rng.integers(4), rng.random() < 0.5
            arrs = [np.rot90(a, k, (1, 2)) for a in arrs]
            if flip: arrs = [a[:, :, ::-1] for a in arrs]

        o, t, wt = [np.ascontiguousarray(a) for a in arrs]

        tt = t.astype(np.float32)
        tt[3] /= 255.0

        return dict(image=torch.from_numpy(o[:1].astype(np.float32) / 255),
                    target=torch.from_numpy(tt),
                    soft_fg=torch.from_numpy(o[2:3].astype(np.float32) / 255),
                    weight=torch.from_numpy(wt)) 

In [ ]:
cache = CACHE / "scale1.0"
train_ds = FilamentDS(set_meta[set_meta.split == "train"], obs_meta, cache, train=True)
val_ds   = FilamentDS(set_meta[set_meta.split == "val"],   obs_meta, cache, train=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, drop_last=True)
b = next(iter(train_dl)); print({k: v.shape for k, v in b.items()})